# GFS Wave 2 data quirks

Every surprise the Phase 1 pipeline found in the Global Flourishing Study
Waves 1–2 public release, in one place, with the evidence. Each one is also
encoded as a parser rule, a schema check, or an override — this notebook is
the human-readable index, not the enforcement.

**Ground rules:** cells print aggregates only — never a raw row. Run
`make data` first; the notebook reads `data/raw/` (two cells) and the
pipeline outputs in `data/`.

Sources of truth: the raw CSVs and codebook PDF (never in git),
`data/catalog.json`, the Parquet tables, and `data/validation_report.md`.

In [ ]:
from pathlib import Path

import polars as pl

ROOT = Path.cwd().resolve()
while not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
RAW = DATA / "raw"
assert (DATA / "flourish.duckdb").exists(), "run `make data` first"

respondents = pl.read_parquet(DATA / "parquet" / "respondents.parquet")
long = pl.scan_parquet(DATA / "parquet" / "responses_long.parquet")
catalog = pl.read_parquet(DATA / "parquet" / "variables.parquet")
value_labels = pl.read_parquet(DATA / "parquet" / "value_labels.parquet")

## 1. Missing values are a single space, and every column reads as text

No cell is empty and none says `NA`: a respondent outside a wave has `" "`
(one space) in that wave's columns. Any reader that trusts type inference
sees object/string columns; the pipeline reads everything as strings and
casts per the catalog (`ingest.py`).

In [ ]:
wave_y2 = pl.read_csv(
    RAW / "gfs_all_countries_wave2_with_midyear.csv",
    columns=["WAVE_Y2"],
    infer_schema=False,
)["WAVE_Y2"]
wave_y2.value_counts().sort("WAVE_Y2").with_columns(
    pl.col("WAVE_Y2").map_elements(repr, return_dtype=pl.String)
)

## 2. Wave flags don't match the codebook

The codebook says every wave flag is coded `1`. The data code `WAVE_Y2 = 2`
and `WAVE_MY = 11` when present (see the cell above and this one). The
pipeline never reads the literal — non-blank means present.

In [ ]:
pl.read_csv(
    RAW / "gfs_all_countries_wave2_with_midyear.csv",
    columns=["WAVE_MY"],
    infer_schema=False,
)["WAVE_MY"].value_counts().sort("WAVE_MY").with_columns(
    pl.col("WAVE_MY").map_elements(repr, return_dtype=pl.String)
)

## 3. The midyear survey was administered two ways — sometimes both in one country

`MIDYEAR_TYPE_MY` = 1 is a standalone interview; 2 means the midyear items
were appended to the Wave 2 interview (same-day answers, so midyear→Y2
"change" is only meaningful for type 1 — proposal §3.2). The proposal reads
as one mode per country; **seven countries actually ran both**.

In [ ]:
modes = (
    respondents.group_by("country_name")
    .agg(
        (pl.col("midyear_type") == 1).sum().alias("separate"),
        (pl.col("midyear_type") == 2).sum().alias("combined"),
    )
    .sort("country_name")
)
both = modes.filter((pl.col("separate") > 0) & (pl.col("combined") > 0))
print(f"countries running BOTH midyear modes: {both.height}")
both

## 4. Sentinel codes are per-variable — a global 98/99 rule corrupts AGE

Ordinary items use −98 / 98 / 99, `AGE` uses −998 / 998 / 999, and
country-specific variables use −9998 / 9998 / 9999. For `AGE`, **99 is a
valid top-code (99+) and 98 is a real age**. The catalog derives each
variable's sentinel set from its own codebook value labels.

In [ ]:
ages = respondents["age"]
print(f"respondents aged 98 (a real age): {(ages == 98).sum()}")
print(f"respondents aged 99 (top-code 99+): {(ages == 99).sum()}")
print(f"age range after cleaning: {ages.min()}–{ages.max()}")
catalog.filter(pl.col("name") == "AGE").select(
    "name",
    "scale_type",
    "min",
    "max",
    "nonresponse_skipped",
    "nonresponse_dk",
    "nonresponse_refused",
)

## 5. Parenthesised labels that are answers, not non-response

Non-response codes are printed in parentheses — `(Saw, skipped)`, `(DK)`,
`(Refused)`, plus the variants `(RF)`, `(Does NOT know household income)`
and `(Refused to give household income)`. But three parenthesised labels
are substantive answers and carry explicit rulings in the overrides:

- `INCOME 9900 = (None/No household income)` — 8,612 respondents at Y1;
- `SELFID2 9997 = (No other response)` — 124,590 respondents with no
  second identity;
- `97 = (Does not apply)` on seven childhood variables (no such
  person/experience).

In [ ]:
value_labels.filter(
    (pl.col("is_nonresponse") == False)  # noqa: E712 — polars expression
    & pl.col("label").str.starts_with("(")
).select("variable", "wave", "code", "label")

## 6. The US file is a subset with 15 extra columns — and 11 missing ones

Every US `ID` is in the global file (exactly the `COUNTRY = 22` rows), and
all 242 shared columns are cell-for-cell identical — so the pipeline loads
the US file only for its extras. It **lacks 11 columns** the global file
has, so any equality check must run on the intersection.

In [ ]:
import csv

with open(RAW / "gfs_all_countries_wave2_with_midyear.csv", newline="") as fh:
    global_cols = next(csv.reader(fh))
with open(RAW / "gfs_us-state-weight_wave2_with-midyear.csv", newline="") as fh:
    us_cols = next(csv.reader(fh))

print(f"global: {len(global_cols)} columns, US: {len(us_cols)} columns")
print(f"missing from the US file ({len(set(global_cols) - set(us_cols))}):")
print(sorted(set(global_cols) - set(us_cols)))
print(f"US-only ({len(set(us_cols) - set(global_cols))}):")
print(sorted(set(us_cols) - set(global_cols)))

## 7. US geography is strings: zero-padded FIPS and pooled states

`FIPS_Y1/Y2` are 5-digit zero-padded county codes — read them as numbers
and `06037` (Los Angeles) becomes `6037`. `STATE_FOR_ANALYSIS_*` has 41
levels: 37 two-letter states plus four pooled groups so small-state
estimates keep adequate n. State weights are blank exactly where the state
is blank (157 respondents at Y1).

In [ ]:
us = respondents.filter(pl.col("country_code") == 22)
fips = us["fips"].drop_nulls()
print(
    f"FIPS: {fips.n_unique()} distinct, all length {fips.str.len_chars().unique().to_list()}, "
    f"{(fips.str.starts_with('0')).sum():,} with a leading zero"
)
levels = us["state"].drop_nulls().unique().sort()
print(f"state levels: {levels.len()}, blank: {us['state'].is_null().sum()}")
print(f"pooled groups: {levels.filter(levels.str.contains('_')).to_list()}")

## 8. Income bands are country-specific — and three countries changed them at Y2

Codes are `country × 100 + band`, with band counts from 9 (Egypt Y1) to 18
(China). An 8-pt footnote in the codebook (dropped by most text
extractors) explains that inflation forced new Argentina and Türkiye bands
at Y2 and Egypt moved to finer brackets — so `INCOME` value labels are per
wave, and **Türkiye's Y2 bands skip 1914/1915**.

In [ ]:
income = value_labels.filter(
    (pl.col("variable") == "INCOME") & pl.col("country_code").is_not_null()
)
bands = (
    income.group_by("wave", "country_code")
    .agg(pl.len().alias("bands"), pl.col("code").max().alias("max_code"))
    .sort("country_code", "wave")
)
print(
    "countries with a Y2-specific band list:",
    bands.filter(pl.col("wave") == "Y2")["country_code"].to_list(),
)
turkey_y2 = income.filter((pl.col("country_code") == 19) & (pl.col("wave") == "Y2"))["code"]
gaps = sorted(set(range(1901, 1917)) - set(turkey_y2.to_list()))
print(f"Türkiye Y2 band codes missing from the codebook list: {gaps}")
bands.filter(pl.col("country_code").is_in([1, 4, 19]))

## 9. Codebook headings that don't match CSV columns

169 unique base column names, but 172 printed codebook headings. The
reconciliation: `ANNUAL_WEIGHT_1` → `ANNUAL_WEIGHT_C1`; `DOI_ANNUAL_Y2`
printed **twice** (the second, labelled "…Midyear Survey", is really
`DOI_MY`); `INCOME_Y1`/`INCOME_Y2` are two entries for one column;
`SELFID1 & SELFID2` is one joint entry for two columns; `FULL_PARTIAL` and
the wave flags cover several columns each. The catalog's alias table
records every mapping.

In [ ]:
import json as _json

cat = _json.loads((DATA / "catalog.json").read_text())
aliases = pl.DataFrame(cat["aliases"]).drop("pages")
print(f"codebook entries mapped: {aliases.height}")
aliases.filter(pl.col("heading") != pl.col("variable"))

## 10. One entry uses a different value-label format entirely (REL9)

Every entry separates codes with `=` (or an en dash for two system
variables) — except `REL9`, which prints `-98. (Saw, skipped)` and then
bare `1 Jodo sect (Honen)` with no separator at all. The parser accepts
the bare form only while a value list is already open, so question wording
that happens to start with a number ("30 minutes or more…") can't be
swallowed.

In [ ]:
value_labels.filter(pl.col("variable") == "REL9").select("code", "label").head(6)

## 11. Other one-offs the validation suite pins

- **One fractional cell in 52.6M**: `DRINKS_Y2 = "4.5"` for a single
  respondent — floored to 4 under an exact-count allowlist.
- **Tanzania `REGION1 = 1832`** (9 respondents at Y2) is missing from the
  codebook, which stops at 1831 — documented via `extra_value_labels`.
- **IDs are 11 or 12 digits** (the country-code prefix is 1–2 digits), not
  uniformly 12.
- The proposal counts **16 substantive midyear items; the file has 15**.
- Interview dates run wider than the proposal's fieldwork windows:
  recruitment from March 2022, Y1 annual interviews into April 2024.
- The PHQ-2/GAD-2 items are coded 1 = nearly every day … **4 = not at
  all** — reversed from standard PHQ scoring; the derive stage scores
  `4 − code`.

In [ ]:
drinks = long.filter((pl.col("variable") == "DRINKS") & (pl.col("wave") == "Y2")).collect()
print(
    f"DRINKS at Y2: n = {drinks.height:,}, values equal to 4: "
    f"{(drinks['value'] == 4).sum():,} (one of them was 4.5 in the raw file)"
)
t1832 = long.filter((pl.col("variable") == "REGION1") & (pl.col("value") == 1832)).collect()
print(f"REGION1 = 1832 rows: {t1832.height}")
ids = respondents["id"].cast(pl.String).str.len_chars()
print(f"ID lengths: {sorted(ids.unique().to_list())}")
print(
    f"midyear substantive items in responses_long: "
    f"{long.filter(pl.col('wave') == 'MY').select('variable').unique().collect().height}"
)
print(
    f"interview dates: recruit from {respondents['doi_recruit_y1'].min()}, "
    f"Y1 annual to {respondents['doi_annual_y1'].max()}"
)

## 12. Retention and midyear coverage vary enormously by country

The proposal's §3.4 numbers, reproduced from the built tables (also in
`data/validation_report.md`). Wave 2 and midyear views must show coverage
next to every estimate, and change figures must use the longitudinal
weights.

In [ ]:
(
    respondents.group_by("country_name")
    .agg(
        pl.len().alias("n_y1"),
        (100 * pl.col("retained_y2").mean()).round(1).alias("retained_pct"),
        (100 * pl.col("has_midyear").mean()).round(1).alias("midyear_pct"),
    )
    .sort("retained_pct", descending=True)
)

## 13. The rectangular panel weight `ANNUAL_WEIGHT_R2` is populated for everyone

Every other post-Wave-1 weight is null exactly where its population ends
(`w_c2`/`w_l2` for the non-retained, `w_l1m` for those without a midyear
interview). `w_r2` is different: it is non-null for **all 207,919 rows** —
including the 79,051 respondents who have no Wave 2 interview at all — and
differs from both `w_c1` and `w_l2` on every row, so it is a genuinely
distinct weight, not a copy. "Weight is non-null" is therefore never a
valid eligibility test. Row eligibility must always come from the flags
(`retained_y2`, `has_midyear`, `midyear_type`); the wave→weight→eligibility
table in `flourish_stats.weights` is the single place that encodes this
(see `docs/METHODS.md`).


In [ ]:
weights_by_retention = (
    respondents.group_by("retained_y2")
    .agg(
        pl.len().alias("rows"),
        pl.col("w_c2").is_not_null().sum().alias("w_c2_non_null"),
        pl.col("w_l2").is_not_null().sum().alias("w_l2_non_null"),
        pl.col("w_r2").is_not_null().sum().alias("w_r2_non_null"),
    )
    .sort("retained_y2")
)
print(weights_by_retention)
print(
    "rows where w_r2 == w_c1:",
    respondents.filter(pl.col("w_r2") == pl.col("w_c1")).height,
)
print(
    "retained rows where w_r2 == w_l2:",
    respondents.filter(pl.col("retained_y2") & (pl.col("w_r2") == pl.col("w_l2"))).height,
)